In [1]:
import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

In [2]:
# Activate Conda Env "Ultra" As the Kernel
import torch
import pandas as pd
from explainers.data_util import batched_k_hop_subgraph
import networkx as nx
from pyvis.network import Network

In [3]:
dataset = torch.load('/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/wn18rr_dataset.pt')
id2entity, id2relation = torch.load('/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/wn18rr_id2name.pt')

In [4]:
split = 'valid'
ratio = 50

if split == 'valid':
    data_index = 1
if split == 'test':
    data_index = 2

root_dir = "/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/docs/RAWExplainer/NBFNet/WN18RR/RAWExplainer-Hyp-Jub1_5/"
output_file = f"{root_dir}/{split}_output_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
explanation_file = f"{root_dir}/{split}_explanations_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
explanations = torch.load(explanation_file, map_location=torch.device('cpu'))
outputs = torch.load(output_file)
output_df = pd.DataFrame(outputs)


pg_root_dir = "/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/docs/PGExplainer/NBFNet/WN18RR/finetune_final_final"
pg_output_file = f"{pg_root_dir}/{split}_output_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
pg_explanation_file = f"{pg_root_dir}/{split}_explanations_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
pg_explanations = torch.load(pg_explanation_file, map_location=torch.device('cpu'))
pg_outputs = torch.load(pg_output_file)
pg_output_df = pd.DataFrame(pg_outputs)

data = dataset[data_index]

print(output_df.iloc[2079], "\n")
print(pg_output_df.iloc[1759], "\n")
print(pg_output_df.iloc[2479], "\n")

Ranking      41099
Heads         4516
Tails        40385
Rel              3
Mode             0
Num_Edges       50
Num_Nodes       26
Inclusion        0
Name: 2079, dtype: int64 

Ranking      16411
Heads         4516
Tails        40385
Rel              3
Mode             0
Num_Edges       50
Num_Nodes       33
Inclusion        0
Name: 1759, dtype: int64 

Ranking      30578
Heads        18594
Tails        29346
Rel              9
Mode             0
Num_Edges       50
Num_Nodes       24
Inclusion        0
Name: 2479, dtype: int64 



In [5]:
def get_row_h_r_t_custom(index, df, id2entity=id2entity, id2relation=id2relation):
    """
    Retrieve head, relation, and tail info for a given index from output DataFrame.

    Args:
        index (int): Row index in the output DataFrame.
        df (pd.DataFrame): DataFrame containing 'Heads', 'Rels', 'Tails' columns.
        id2entity (dict, optional): Mapping from entity ID → name.
        id2relation (dict, optional): Mapping from relation ID → name.

    Returns:
        tuple: (row, head_name, rel_name, tail_name, head_id, tail_id)
    """
    row = df.iloc[index]
    
    # Extract IDs
    head_id = int(row["Heads"])
    tail_id = int(row["Tails"])
    rel_id = int(row["Rel"]) if "Rel" in row else int(row["Rels"])  # handle naming differences

    # Convert to names if mappings provided
    if id2entity is not None:
        head_name = id2entity.get(head_id, f"id_{head_id}")
        tail_name = id2entity.get(tail_id, f"id_{tail_id}")
    else:
        head_name, tail_name = head_id, tail_id

    if id2relation is not None:
        rel_name = id2relation.get(rel_id, f"id_{rel_id}")
    else:
        rel_name = rel_id

    return row, head_name, rel_name, tail_name, head_id, tail_id


print("RAW Explainer")
# print(output_df)
print(get_row_h_r_t_custom(2079, output_df))

print("\n\nPG Explainer")
# print(pg_output_df)
print(get_row_h_r_t_custom(2479, pg_output_df))


RAW Explainer
(Ranking      41099
Heads         4516
Tails        40385
Rel              3
Mode             0
Num_Edges       50
Num_Nodes       26
Inclusion        0
Name: 2079, dtype: int64, 'big_shot.n.01', '_hypernym', 'supremo.n.01', 4516, 40385)


PG Explainer
(Ranking      30578
Heads        18594
Tails        29346
Rel              9
Mode             0
Num_Edges       50
Num_Nodes       24
Inclusion        0
Name: 2479, dtype: int64, 'chemistry.n.01', '_synset_domain_topic_of', 'carbonize.v.01', 18594, 29346)


In [6]:
def get_row_h_r_t(index, output_file_df):
    row = output_file_df.iloc[index]
    if row['Mode'] == 1:
        rel = id2relation[row['Rel']]
    else:
        rel = id2relation[row['Rel']]+'_inv'
    head_id = row['Heads']
    tail_id = row['Tails']
    head = id2entity[head_id]
    tail = id2entity[tail_id]
    return row, head, rel, tail, head_id, tail_id

In [7]:
# def vizualize_explanation(index, hop=6):
#     row, head, rel, tail, head_id, tail_id = get_row_h_r_t(index, output_df)
#     print(f"*** Query: {head, rel}. Answer: {tail}, Rank given the explanation: {row['Ranking']} ***")
#     nodes, edges = batched_k_hop_subgraph(torch.tensor([head_id]).unsqueeze(0), hop, data.edge_index, data.num_nodes)

#     print(f'Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}')
#     edges_mask = edges.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
#     full_edge_index = data.edge_index[:, edges_mask]

#     expl = explanations[index]
    
#     # Normalize explanation into indices relative to full_edge_index
#     if expl.dtype == torch.bool:
#         if expl.numel() == full_edge_index.shape[1]:
#             sub_mask = expl.to(device=full_edge_index.device)
#             expl_edge_index = full_edge_index[:, sub_mask]
#         elif expl.numel() == data.edge_index.shape[1]:
#             global_mask = expl.to(device=data.edge_index.device)
#             combined_mask = torch.logical_and(global_mask, edges_mask)
#             expl_edge_index = data.edge_index[:, combined_mask]
#         else:
#             raise ValueError(f"Unexpected boolean explanation length {expl.numel()} not matching subgraph ({full_edge_index.shape[1]}) or global ({data.edge_index.shape[1]}).")
#     else:
#         # Assume list of indices into the subgraph edges
#         expl_indices = expl.to(dtype=torch.long, device=full_edge_index.device)
#         valid = (expl_indices >= 0) & (expl_indices < full_edge_index.shape[1])
#         expl_indices = expl_indices[valid]
#         expl_edge_index = full_edge_index.index_select(1, expl_indices)

#     print(f'Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}')
#     print(f'Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}')

#     # Create a directed graph
#     G = nx.DiGraph()

#     # Add nodes - convert to int to ensure compatibility with pyvis
#     head_id_int = int(head_id)
#     tail_id_int = int(tail_id)
    
#     G.add_node(head_id_int, color='#FF0000', size=25, title=f"HEAD: {head}")
#     G.add_node(tail_id_int, color='#00FF00', size=25, title=f"TAIL: {tail}")

#     # Add edges from expl_edge_index
#     for i in range(expl_edge_index.shape[1]):
#         source = int(expl_edge_index[0, i].item())
#         target = int(expl_edge_index[1, i].item())
#         # G.add_edge(source, target,arrows='to')
#         G.add_edge(source, target)

#     attrs = {head_id_int: "#FF0000", tail_id_int: "#00FF00"}
#     nx.set_node_attributes(G, attrs, name='color')

#     try:
#         paths = list(nx.all_simple_paths(G, source=head_id_int, target=tail_id_int, cutoff=6))
    
#         attrs = {}
#         if len(paths) > 0:
#             connected = True
#         else:
#             connected = False
    
#         print(f'Is the tail connected to the head? {connected}')
#         for path in paths:
#             src = head_id_int
#             for node in path[1:]:
#                 edge = (src, node)
#                 attrs[edge] = {'color':"#b27ebd"}
#                 src = node

#         nx.set_edge_attributes(G, attrs)
#     except:
#         print(f'Is the tail connected to the head? {False}')
            
    
#     return G, row['Ranking']
    

In [8]:
# # Control the index (row number of output_df of the instance you want to inspect here)
# # index = 588
# # indices_to_plot = [588, 3148, 4302]
# indices_to_plot = [1759]

In [9]:
# for idx in indices_to_plot:
#     G, ranking = vizualize_explanation(idx, hop=10)

In [10]:
# # *** Explanation Graph Visualization ***
# # RED NODE: The Query Head (if inside the explanation)
# # GREEN NODE: The Answer Tail (if inside the explanation)
# # Purple Edges: Edges that are in any path that connected Red to Green
# for index in indices_to_plot:
#     net = Network(notebook = True, cdn_resources = "remote",
#                         bgcolor = "#222222",
#                         font_color = "white",
#                         height = "750px",
#                         width = "100%",
#                         select_menu = True,
#                         filter_menu = True,
#                         directed=True
#     )
#     net.from_nx(G)
#     net.inherit_edge_colors(False)
#     net.set_edge_smooth('dynamic')
#     os.makedirs('./plots/PGExplainer', exist_ok=True)
#     net.show(f'./plots/PGExplainer/graph_wn_pg_{index}.html')

## Explanation Visualization

In [ ]:
import numpy as np
from igraph import Graph
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
import pickle
from notebooks.vizualize import prepare_expl
from nbfnet.tasks import edge_match


def save_layout(coords, name_to_idx, filename):
    """Save layout to disk"""
    layout_data = {
        'coords': coords,
        'name_to_idx': name_to_idx
    }
    with open(filename, 'wb') as f:
        pickle.dump(layout_data, f)
    print(f"Layout saved to {filename}")

def load_layout(filename):
    """Load layout from disk"""
    with open(filename, 'rb') as f:
        layout_data = pickle.load(f)
    print(f"Layout loaded from {filename}")
    return layout_data

def compute_unified_layout_for_both(index1, output_df1, explanations1, 
                                     index2, output_df2, explanations2,
                                     data, hop=10, seed=9999):
    """
    Compute unified layout combining subgraphs from both indices.
    """
    all_edges = set()
    
    # Process first graph
    row1, head1, rel1, tail1, head_id1, tail_id1 = get_row_h_r_t(index1, output_df1)
    nodes1, edges1 = batched_k_hop_subgraph(
        torch.tensor([head_id1]).unsqueeze(0), hop, data.edge_index, data.num_nodes
    )
    edges_mask1 = edges1.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
    full_edge_index1 = data.edge_index[:, edges_mask1]
    
    for i in range(full_edge_index1.shape[1]):
        src = int(full_edge_index1[0, i].item())
        tgt = int(full_edge_index1[1, i].item())
        all_edges.add((src, tgt))
    
    # Process second graph
    row2, head2, rel2, tail2, head_id2, tail_id2 = get_row_h_r_t(index2, output_df2)
    nodes2, edges2 = batched_k_hop_subgraph(
        torch.tensor([head_id2]).unsqueeze(0), hop, data.edge_index, data.num_nodes
    )
    edges_mask2 = edges2.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
    full_edge_index2 = data.edge_index[:, edges_mask2]
    
    for i in range(full_edge_index2.shape[1]):
        src = int(full_edge_index2[0, i].item())
        tgt = int(full_edge_index2[1, i].item())
        all_edges.add((src, tgt))
    
    print(f"Computing unified layout with {len(all_edges)} edges")
    
    # Build combined graph
    combined_edge_list = list(all_edges)
    g_combined = Graph.TupleList(combined_edge_list, directed=True)
    
    # Compute layout
    np.random.seed(seed)
    layout = g_combined.layout_fruchterman_reingold()
    coords = np.array(layout.coords)
    name_to_idx = {int(g_combined.vs[i]['name']): i for i in range(len(g_combined.vs))}
    
    print(f"Layout computed with {len(name_to_idx)} nodes")
    
    return {'coords': coords, 'name_to_idx': name_to_idx}

def plot_single_graph(ax, index, output_df, expl, data, hop, precomputed_layout, title, dataset=dataset):
    """
    Plot a single graph on the given axis.
    """
    # row, head, rel, tail, head_id, tail_id = get_row_h_r_t(index, output_df)
    # print(f"*** {title} - Query: ({head}, {rel}). Answer: {tail}, Rank: {row['Ranking']} ***")
    
    # # Get full k-hop subgraph
    # nodes, edges = batched_k_hop_subgraph(
    #     torch.tensor([head_id]).unsqueeze(0), hop, data.edge_index, data.num_nodes
    # )
    # edges_mask = edges.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
    # full_edge_index = data.edge_index[:, edges_mask]
    
    # expl = explanations[index]
    
    # # Normalize explanation
    # if expl.dtype == torch.bool:
    #     if expl.numel() == full_edge_index.shape[1]:
    #         expl_edge_index = full_edge_index[:, expl.to(device=full_edge_index.device)]
    #     elif expl.numel() == data.edge_index.shape[1]:
    #         combined_mask = torch.logical_and(expl.to(device=data.edge_index.device), edges_mask)
    #         expl_edge_index = data.edge_index[:, combined_mask]
    #     else:
    #         raise ValueError(f"Unexpected boolean explanation length")
    # else:
    #     expl_indices = expl.to(dtype=torch.long, device=full_edge_index.device)
    #     valid = (expl_indices >= 0) & (expl_indices < full_edge_index.shape[1])
    #     expl_edge_index = full_edge_index.index_select(1, expl_indices[valid])

    num_relations = dataset.num_relations

    # The GNN_eval Result
    row = output_df.iloc[index]
    if row["Mode"] == 1:
        rel = id2relation[row["Rel"]]
    else:
        rel = id2relation[row["Rel"]] + "_inv"
    head_id = row["Heads"]
    tail_id = row["Tails"]
    rel_id = row["Rel"]
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    # Print the ranking
    print(f"*** Query: {head, rel}. Answer: {tail} ***")

    nodes, edges = batched_k_hop_subgraph(
        torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes
    )

    tail_in_6hop = nodes[0, tail_id].item()
    print(
        f"Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}"
    )
    full_edge_index = data.edge_index[:, edges.squeeze()]

    if row["Mode"] == 1:
        query = torch.tensor([[head_id, rel_id]])
    else:
        query = torch.tensor([[head_id, rel_id + num_relations // 2]])
    # find the matching explanations
    all_query = expl[0]
    row_id, num_match = edge_match(all_query.T, query.T)
    assert torch.all(num_match == 1)
    expl = expl[1][row_id]

    # expl = explanations[index]
    # query = expl[:2]
    # expl = expl[2:]
    # check if query matches the output.
    # Remove any non-edges:
    non_edge_mask = expl < 0
    expl = expl[~non_edge_mask]
    # Check that the explanation are restricted only to within the ego_network (not necessary)
    # assert torch.all(torch.logical_and(expl, ~edges.squeeze()) == False)
    expl_edge_index = data.edge_index[:, expl]
    expl_edge_type = data.edge_type[expl]

    head_in_expl = torch.any((expl_edge_index == head_id)).item()
    tail_in_expl = torch.any((expl_edge_index == tail_id)).item()
    print(
        f"Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}"
    )
    print(
        f"Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}"
    )

    # Create a directed graph
    G = nx.MultiDiGraph()

    # Add edges from expl_edge_index
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        relation = expl_edge_type[i].item()
        if relation < num_relations // 2:
            rel_name = id2relation[relation]
        else:
            rel_name = id2relation[relation - num_relations // 2] + "_inv"
        # G.add_edge(source, target,arrows='to')
        G.add_edge(source, target, title=rel_name)
    
    head_id_int = int(head_id)
    tail_id_int = int(tail_id)
    
    # Get explanation edges and nodes
    expl_edges = {(int(expl_edge_index[0, i].item()), int(expl_edge_index[1, i].item())) 
                  for i in range(expl_edge_index.shape[1])}
    
    # Get nodes that are part of explanation
    expl_nodes = set()
    for src, tgt in expl_edges:
        expl_nodes.add(src)
        expl_nodes.add(tgt)
    
    # Always include head and tail nodes
    expl_nodes.add(head_id_int)
    expl_nodes.add(tail_id_int)
    
    # Build full graph for background
    full_edge_list = [(int(full_edge_index[0, i].item()), int(full_edge_index[1, i].item())) 
                      for i in range(full_edge_index.shape[1])]
    
    # Use precomputed layout
    coords = precomputed_layout['coords']
    name_to_idx = precomputed_layout['name_to_idx']
    
    # Get all node IDs
    all_node_ids = set(name_to_idx.keys())
    
    ax.set_facecolor('white')
    
    # Draw background edges (all edges not in explanation)
    bg_segs = []
    for src, tgt in full_edge_list:
        if (src, tgt) not in expl_edges:
            if src in name_to_idx and tgt in name_to_idx:
                seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
                bg_segs.append(seg)
    
    if bg_segs:
        lc_bg = LineCollection(bg_segs, colors='#DDDDDD', alpha=0.15, linewidths=0.3, zorder=1)
        ax.add_collection(lc_bg)
    
    # Draw explanation edges
    expl_segs = []
    for src, tgt in list(expl_edges):
        if src in name_to_idx and tgt in name_to_idx:
            seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
            expl_segs.append(seg)
    
    if expl_segs:
        lc_expl = LineCollection(expl_segs, colors='#0066FF', alpha=0.8, linewidths=1.5, zorder=3)
        ax.add_collection(lc_expl)
    
    # Draw background nodes (not in explanation)
    bg_node_ids = all_node_ids - expl_nodes
    if bg_node_ids:
        bg_coords = np.array([coords[name_to_idx[nid]] for nid in bg_node_ids if nid in name_to_idx])
        if len(bg_coords) > 0:
            ax.scatter(bg_coords[:, 0], bg_coords[:, 1], 
                      c='#CCCCCC', s=15, zorder=2, alpha=0.3, edgecolors='none')
    
    # Draw explanation nodes (light blue)
    expl_only_nodes = expl_nodes - {head_id_int, tail_id_int}
    if expl_only_nodes:
        expl_coords = np.array([coords[name_to_idx[nid]] for nid in expl_only_nodes if nid in name_to_idx])
        if len(expl_coords) > 0:
            ax.scatter(expl_coords[:, 0], expl_coords[:, 1], 
                      c='#87CEEB', s=100, zorder=4, edgecolors='white', linewidths=1)
    
    # Draw head (red)
    if head_id_int in name_to_idx:
        hx, hy = coords[name_to_idx[head_id_int]]
        ax.scatter(hx, hy, c='#FF0000', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    # Draw tail (green)
    if tail_id_int in name_to_idx:
        tx, ty = coords[name_to_idx[tail_id_int]]
        ax.scatter(tx, ty, c='#00FF00', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')
    ax.autoscale()
    
    return row['Ranking']

def visualize_comparison_side_by_side(index1, output_df1, explanations1,
                                       index2, output_df2, explanations2,
                                       data, hop=10, seed=9999, 
                                       save_path='comparison.png',
                                       title1='RAW Explainer', 
                                       title2='PG Explainer'):
    """
    Create side-by-side visualization comparing two explanations.
    
    Parameters:
    - index1: Index in output_df1
    - output_df1: First output DataFrame
    - explanations1: First explanations
    - index2: Index in output_df2
    - output_df2: Second output DataFrame
    - explanations2: Second explanations
    - data: Graph data
    - hop: Number of hops
    - seed: Random seed
    - save_path: Where to save the comparison image
    - title1: Title for first plot
    - title2: Title for second plot
    """
    # Compute unified layout
    unified_layout = compute_unified_layout_for_both(
        index1, output_df1, explanations1,
        index2, output_df2, explanations2,
        data, hop, seed
    )
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 12), facecolor='white', dpi=150)
    
    # Plot first graph
    ranking1 = plot_single_graph(ax1, index1, output_df1, explanations1, data, hop, 
                                  unified_layout, title1)
    
    # Plot second graph
    ranking2 = plot_single_graph(ax2, index2, output_df2, explanations2, data, hop, 
                                  unified_layout, title2)
    
    # Add shared legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF0000', 
               markersize=12, label='Head (Query)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#00FF00', 
               markersize=12, label='Tail (Answer)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#87CEEB', 
               markersize=8, label='Explanation Nodes', markeredgecolor='white', markeredgewidth=1),
        Line2D([0], [0], color='#0066FF', linewidth=2, label='Explanation Edges'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#CCCCCC', 
               markersize=5, label='Background Nodes', alpha=0.5),
        Line2D([0], [0], color='#DDDDDD', linewidth=1, label='Background Edges', alpha=0.3),
    ]
    
    fig.legend(handles=legend_elements, loc='upper center', ncol=6, 
               framealpha=0.9, fontsize=12, bbox_to_anchor=(0.5, 0.98))
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, facecolor='white', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\nComparison saved to: {save_path}")
    print(f"{title1} Ranking: {ranking1}")
    print(f"{title2} Ranking: {ranking2}")
    
    return ranking1, ranking2, unified_layout


# ===== USAGE =====

# Example 1: Compare same query at different indices
raw_index = 65
pg_index = 130

expl_pg = prepare_expl(pg_explanations)
expl_raw = prepare_expl(explanations)

seed_val=1000
visualize_comparison_side_by_side(
    raw_index, output_df, expl_raw,
    pg_index, pg_output_df, expl_pg,
    data, 
    hop=5230, 
    seed=seed_val,
    save_path=f'./plots/comparison_raw_vs_pg_final_new_new_{raw_index}_{pg_index}_re_seed{seed_val}.png',
    title1='RAW Explainer',
    title2='PG Explainer'
)

In [ ]:
import numpy as np
from igraph import Graph
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
import pickle
from notebooks.vizualize import prepare_expl
from nbfnet.tasks import edge_match


def save_layout(coords, name_to_idx, filename):
    """Save layout to disk"""
    layout_data = {
        'coords': coords,
        'name_to_idx': name_to_idx
    }
    with open(filename, 'wb') as f:
        pickle.dump(layout_data, f)
    print(f"Layout saved to {filename}")

def load_layout(filename):
    """Load layout from disk"""
    with open(filename, 'rb') as f:
        layout_data = pickle.load(f)
    print(f"Layout loaded from {filename}")
    return layout_data

def compute_unified_layout_for_both(index1, output_df1, explanations1, 
                                     index2, output_df2, explanations2,
                                     data, hop=10, seed=9999):
    """
    Compute unified layout combining subgraphs from both indices.
    Uses tighter layout parameters for more compact visualization.
    """
    all_edges = set()
    
    # Process first graph
    row1, head1, rel1, tail1, head_id1, tail_id1 = get_row_h_r_t(index1, output_df1)
    nodes1, edges1 = batched_k_hop_subgraph(
        torch.tensor([head_id1]).unsqueeze(0), hop, data.edge_index, data.num_nodes
    )
    edges_mask1 = edges1.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
    full_edge_index1 = data.edge_index[:, edges_mask1]
    
    for i in range(full_edge_index1.shape[1]):
        src = int(full_edge_index1[0, i].item())
        tgt = int(full_edge_index1[1, i].item())
        all_edges.add((src, tgt))
    
    # Process second graph
    row2, head2, rel2, tail2, head_id2, tail_id2 = get_row_h_r_t(index2, output_df2)
    nodes2, edges2 = batched_k_hop_subgraph(
        torch.tensor([head_id2]).unsqueeze(0), hop, data.edge_index, data.num_nodes
    )
    edges_mask2 = edges2.squeeze().to(dtype=torch.bool, device=data.edge_index.device)
    full_edge_index2 = data.edge_index[:, edges_mask2]
    
    for i in range(full_edge_index2.shape[1]):
        src = int(full_edge_index2[0, i].item())
        tgt = int(full_edge_index2[1, i].item())
        all_edges.add((src, tgt))
    
    print(f"Computing unified layout with {len(all_edges)} edges")
    
    # Build combined graph
    combined_edge_list = list(all_edges)
    g_combined = Graph.TupleList(combined_edge_list, directed=True)
    
    # Compute layout with tighter parameters
    np.random.seed(seed)
    # Use smaller area for tighter clustering
    num_nodes = len(g_combined.vs)
    layout = g_combined.layout_fruchterman_reingold(
        niter=500,  # More iterations for better convergence
        grid='nogrid'  # Disable grid for more natural clustering
    )
    
    coords = np.array(layout.coords)
    name_to_idx = {int(g_combined.vs[i]['name']): i for i in range(len(g_combined.vs))}
    
    print(f"Layout computed with {len(name_to_idx)} nodes")
    
    return {'coords': coords, 'name_to_idx': name_to_idx}

def plot_single_graph(ax, index, output_df, expl, data, hop, precomputed_layout, title, dataset=dataset):
    """
    Plot a single graph on the given axis with option to filter distant background nodes.
    """
    num_relations = dataset.num_relations

    # The GNN_eval Result
    row = output_df.iloc[index]
    if row["Mode"] == 1:
        rel = id2relation[row["Rel"]]
    else:
        rel = id2relation[row["Rel"]] + "_inv"
    head_id = row["Heads"]
    tail_id = row["Tails"]
    rel_id = row["Rel"]
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    # Print the ranking
    print(f"*** Query: {head, rel}. Answer: {tail} ***")

    nodes, edges = batched_k_hop_subgraph(
        torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes
    )

    tail_in_6hop = nodes[0, tail_id].item()
    print(
        f"Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}"
    )
    full_edge_index = data.edge_index[:, edges.squeeze()]

    if row["Mode"] == 1:
        query = torch.tensor([[head_id, rel_id]])
    else:
        query = torch.tensor([[head_id, rel_id + num_relations // 2]])
    # find the matching explanations
    all_query = expl[0]
    row_id, num_match = edge_match(all_query.T, query.T)
    assert torch.all(num_match == 1)
    expl = expl[1][row_id]

    # Remove any non-edges:
    non_edge_mask = expl < 0
    expl = expl[~non_edge_mask]
    expl_edge_index = data.edge_index[:, expl]
    expl_edge_type = data.edge_type[expl]

    head_in_expl = torch.any((expl_edge_index == head_id)).item()
    tail_in_expl = torch.any((expl_edge_index == tail_id)).item()
    print(
        f"Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}"
    )
    print(
        f"Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}"
    )

    # Create a directed graph
    G = nx.MultiDiGraph()

    # Add edges from expl_edge_index
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        relation = expl_edge_type[i].item()
        if relation < num_relations // 2:
            rel_name = id2relation[relation]
        else:
            rel_name = id2relation[relation - num_relations // 2] + "_inv"
        G.add_edge(source, target, title=rel_name)
    
    head_id_int = int(head_id)
    tail_id_int = int(tail_id)
    
    # Get explanation edges and nodes
    expl_edges = {(int(expl_edge_index[0, i].item()), int(expl_edge_index[1, i].item())) 
                  for i in range(expl_edge_index.shape[1])}
    
    # Get nodes that are part of explanation
    expl_nodes = set()
    for src, tgt in expl_edges:
        expl_nodes.add(src)
        expl_nodes.add(tgt)
    
    # Always include head and tail nodes
    expl_nodes.add(head_id_int)
    expl_nodes.add(tail_id_int)
    
    # Build full graph for background
    full_edge_list = [(int(full_edge_index[0, i].item()), int(full_edge_index[1, i].item())) 
                      for i in range(full_edge_index.shape[1])]
    
    # Use precomputed layout
    coords = precomputed_layout['coords']
    name_to_idx = precomputed_layout['name_to_idx']
    
    # Calculate center of explanation nodes for filtering
    expl_coords = np.array([coords[name_to_idx[nid]] for nid in expl_nodes if nid in name_to_idx])
    if len(expl_coords) > 0:
        center = np.mean(expl_coords, axis=0)
        # Calculate radius as 1.5x the max distance from center to any explanation node
        distances = np.linalg.norm(expl_coords - center, axis=1)
        radius = np.max(distances) * 2.0 if len(distances) > 0 else np.inf
    else:
        center = np.array([0, 0])
        radius = np.inf
    
    # Get all node IDs
    all_node_ids = set(name_to_idx.keys())
    
    # Filter background nodes: only keep those within radius of explanation center
    bg_node_ids = all_node_ids - expl_nodes
    filtered_bg_nodes = set()
    for nid in bg_node_ids:
        if nid in name_to_idx:
            node_coord = coords[name_to_idx[nid]]
            dist = np.linalg.norm(node_coord - center)
            if dist <= radius:
                filtered_bg_nodes.add(nid)
    
    ax.set_facecolor('white')
    
    # Draw background edges (only for filtered nodes)
    bg_segs = []
    for src, tgt in full_edge_list:
        if (src, tgt) not in expl_edges:
            # Only draw if both nodes are in filtered background set or explanation set
            if ((src in filtered_bg_nodes or src in expl_nodes) and 
                (tgt in filtered_bg_nodes or tgt in expl_nodes)):
                if src in name_to_idx and tgt in name_to_idx:
                    seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
                    bg_segs.append(seg)
    
    if bg_segs:
        lc_bg = LineCollection(bg_segs, colors='#DDDDDD', alpha=0.15, linewidths=0.3, zorder=1)
        ax.add_collection(lc_bg)
    
    # Draw explanation edges
    expl_segs = []
    for src, tgt in list(expl_edges):
        if src in name_to_idx and tgt in name_to_idx:
            seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
            expl_segs.append(seg)
    
    if expl_segs:
        lc_expl = LineCollection(expl_segs, colors='#0066FF', alpha=0.8, linewidths=1.5, zorder=3)
        ax.add_collection(lc_expl)
    
    # Draw filtered background nodes
    if filtered_bg_nodes:
        bg_coords = np.array([coords[name_to_idx[nid]] for nid in filtered_bg_nodes])
        if len(bg_coords) > 0:
            ax.scatter(bg_coords[:, 0], bg_coords[:, 1], 
                      c='#CCCCCC', s=15, zorder=2, alpha=0.3, edgecolors='none')
    
    # Draw explanation nodes (light blue)
    expl_only_nodes = expl_nodes - {head_id_int, tail_id_int}
    if expl_only_nodes:
        expl_coords = np.array([coords[name_to_idx[nid]] for nid in expl_only_nodes if nid in name_to_idx])
        if len(expl_coords) > 0:
            ax.scatter(expl_coords[:, 0], expl_coords[:, 1], 
                      c='#87CEEB', s=100, zorder=4, edgecolors='white', linewidths=1)
    
    # Draw head (red)
    if head_id_int in name_to_idx:
        hx, hy = coords[name_to_idx[head_id_int]]
        ax.scatter(hx, hy, c='#FF0000', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    # Draw tail (green)
    if tail_id_int in name_to_idx:
        tx, ty = coords[name_to_idx[tail_id_int]]
        ax.scatter(tx, ty, c='#00FF00', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')
    ax.autoscale()
    
    return row['Ranking']

def visualize_comparison_side_by_side(index1, output_df1, explanations1,
                                       index2, output_df2, explanations2,
                                       data, hop=10, seed=9999, 
                                       save_path='comparison.png',
                                       title1='RAW Explainer', 
                                       title2='PG Explainer'):
    """
    Create side-by-side visualization comparing two explanations.
    
    Parameters:
    - index1: Index in output_df1
    - output_df1: First output DataFrame
    - explanations1: First explanations
    - index2: Index in output_df2
    - output_df2: Second output DataFrame
    - explanations2: Second explanations
    - data: Graph data
    - hop: Number of hops
    - seed: Random seed
    - save_path: Where to save the comparison image
    - title1: Title for first plot
    - title2: Title for second plot
    """
    # Compute unified layout
    unified_layout = compute_unified_layout_for_both(
        index1, output_df1, explanations1,
        index2, output_df2, explanations2,
        data, hop, seed
    )
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(30, 12), facecolor='white', dpi=150)
    
    # Plot first graph
    ranking1 = plot_single_graph(ax1, index1, output_df1, explanations1, data, hop, 
                                  unified_layout, title1)
    
    # Plot second graph
    ranking2 = plot_single_graph(ax2, index2, output_df2, explanations2, data, hop, 
                                  unified_layout, title2)
    
    # Add shared legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF0000', 
               markersize=12, label='Head (Query)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#00FF00', 
               markersize=12, label='Tail (Answer)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#87CEEB', 
               markersize=8, label='Explanation Nodes', markeredgecolor='white', markeredgewidth=1),
        Line2D([0], [0], color='#0066FF', linewidth=2, label='Explanation Edges'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#CCCCCC', 
               markersize=5, label='Background Nodes', alpha=0.5),
        Line2D([0], [0], color='#DDDDDD', linewidth=1, label='Background Edges', alpha=0.3),
    ]
    
    fig.legend(handles=legend_elements, loc='upper center', ncol=6, 
               framealpha=0.9, fontsize=12, bbox_to_anchor=(0.5, 0.98))
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, facecolor='white', bbox_inches='tight', dpi=300)
    plt.close()
    
    print(f"\nComparison saved to: {save_path}")
    print(f"{title1} Ranking: {ranking1}")
    print(f"{title2} Ranking: {ranking2}")
    
    return ranking1, ranking2, unified_layout


# ===== USAGE =====

# Example 1: Compare same query at different indices
raw_index = 65
pg_index = 130

expl_pg = prepare_expl(pg_explanations)
expl_raw = prepare_expl(explanations)

seed_val = 12
visualize_comparison_side_by_side(
    raw_index, output_df, expl_raw,
    pg_index, pg_output_df, expl_pg,
    data, 
    hop=10, 
    seed=seed_val,
    save_path=f'./plots/comparison_raw_vs_pg_final_new_new_{raw_index}_{pg_index}_re_seed{seed_val}.png',
    title1='RAW Explainer',
    title2='PG Explainer'
)